In [5]:
import pandas as pd
import numpy as np
import os

In [9]:
# Load the dataset
df = pd.read_csv('../data/Crop_recommendation_ethiopia_real.csv')

# ---------------------------------------------------------------------------
# 1. ALTITUDE from PS (surface pressure)
# This export stores PS in kPa (~74–84), NOT Pascals (~74_000–84_000).
# Dividing by 100 produced ~0.78 hPa → ~33 km (stratosphere). Fix: kPa → hPa.
# ---------------------------------------------------------------------------
df['PS_hPa'] = df['PS'] * 10  # kPa → hPa
sea_level = 1013.25  # hPa
df['altitude_m'] = 44330 * (1 - (df['PS_hPa'] / sea_level) ** 0.1903)
print("Altitude (m) — expect Ethiopian highlands ~1,200–3,000:")
print(df['altitude_m'].describe())

# Average temperature (°C) across seasonal T2M max/min columns
temp_cols = [col for col in df.columns if 'T2M' in col]
df['temperature'] = df[temp_cols].mean(axis=1)

# ---------------------------------------------------------------------------
# 2. RELATIVE HUMIDITY (%)
# QV2M in this export is already g/kg (~8–12), NOT kg/kg.
# Multiplying by 1000 incorrectly produced ~8,500–11,500.
# Convert specific humidity → RH% using Magnus formula + ambient pressure.
# ---------------------------------------------------------------------------
hum_cols = [col for col in df.columns if 'QV2M' in col]
qv_g_kg = df[hum_cols].mean(axis=1)  # specific humidity in g/kg
q = qv_g_kg / 1000.0  # kg/kg for vapor-pressure formula
T = df['temperature']
e = q * df['PS_hPa'] / (0.622 + 0.378 * q)  # actual vapor pressure (hPa)
e_s = 6.112 * np.exp((17.67 * T) / (T + 243.5))  # saturation vapor pressure
df['humidity'] = ((e / e_s) * 100).clip(35, 95)  # RH % (Ethiopian ag range)
print("\nRelative Humidity (%) — expect ~35–95:")
print(df['humidity'].describe())

# Annual rainfall approximation (sum of seasonal PRECTOTCORR)
rain_cols = [col for col in df.columns if 'PRECTOTCORR' in col]
df['rainfall'] = df[rain_cols].sum(axis=1)

# ---------------------------------------------------------------------------
# 3. NITROGEN unit harmonization
# Raw N is a tiny fraction (~0.0–0.7) while P/K are mg/kg-scale integers.
# Scale N ×1000 → mg/kg (ppm) so it is not invisible to distance-based models
# and fertilizer deficit math stays coherent.
# ---------------------------------------------------------------------------
df['N'] = df['N'] * 1000
print("\nNitrogen (mg/kg) after ×1000:")
print(df['N'].describe())

df['soil_moisture'] = df['GWETTOP']  # Topsoil wetness (0–1)

# ---------------------------------------------------------------------------
# 4. SOIL COLOR — normalize messy labels, one-hot encode, KEEP in features
# ---------------------------------------------------------------------------
def normalize_soil_color(value):
    s = str(value).lower().strip()
    if 'black' in s:
        return 'black'
    if 'red' in s:
        return 'red'
    if 'gray' in s or 'grey' in s:
        return 'gray'
    if 'brown' in s or 'yellow' in s:
        return 'brown'
    return 'other'

df['Soilcolor'] = df['Soilcolor'].map(normalize_soil_color)
df = pd.get_dummies(df, columns=['Soilcolor'], prefix='soilcolor', dtype=int)
df = df.rename(columns={'Ph': 'ph'})

# Dynamically absorb soil-color dummies into the training feature list
soilcolor_cols = [col for col in df.columns if col.startswith('soilcolor_')]
feature_cols = [
    'N', 'P', 'K', 'ph', 'temperature', 'humidity', 'rainfall',
    'altitude_m', 'Zn', 'S', 'soil_moisture',
] + soilcolor_cols

df = df[feature_cols + ['label']]
X = df[feature_cols]
y = df['label']

print("\nEngineered features shape:", X.shape)
print("Feature columns:", feature_cols)
print("Soil color dummies:", soilcolor_cols)
print("Sample:\n", X.head())

# Save processed for modeling
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/engineered.csv', index=False)
X.to_csv('../data/processed/X_engineered.csv', index=False)
pd.DataFrame(y).to_csv('../data/processed/y.csv', index=False)
pd.Series(feature_cols, name='feature').to_csv(
    '../data/processed/feature_cols.csv', index=False
)

print("\nSaved engineered.csv / X_engineered.csv / y.csv / feature_cols.csv")

Altitude sample (m): count     3867.000000
mean     32990.465662
std         65.144229
min      32846.778606
25%      32957.289113
50%      32994.519582
75%      33025.017679
max      33109.445967
Name: altitude_m, dtype: float64

New engineered features shape: (3867, 11)
Sample:
       N       P        K    ph  temperature      humidity   rainfall  \
0  0.23   5.401  738.231  5.81    16.512917  10024.166666  24.916667   
1  0.23  10.478  606.382  5.43    16.512917  10024.166666  24.916667   
2  0.23   6.847  386.580  5.41    16.512917  10024.166666  24.916667   
3  0.23   3.418  207.086  5.65    16.512917  10024.166666  24.916667   
4  0.23  39.282  317.357  5.27    16.512917  10024.166666  24.916667   

     altitude_m        Zn       S  soil_moisture  
0  33028.366005  2.976000  13.816           0.73  
1  33028.366005  3.077000  16.421           0.73  
2  33028.366005  6.611000  16.557           0.73  
3  33028.366005  0.460181  16.075           0.73  
4  33028.366005  2.743000  12.